# 03 - Inference & Submission


In [2]:
import os, sys, subprocess
REPO_URL = "https://github.com/azizahadilaa/document-image-classification-kaggle.git" 
REPO_DIR = "/kaggle/working/document-image-classification-kaggle"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
sys.path.append(REPO_DIR)

import pandas as pd, numpy as np, torch
from scripts.utils import set_seed, find_data_dir
set_seed(42)
DATA_DIR = find_data_dir()
CACHE_DIR = "/kaggle/temp/cache"
IMG_SIZE = 320
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print(DATA_DIR, train_df.shape, test_df.shape)

Cloning into '/kaggle/working/document-image-classification-kaggle'...


/kaggle/input/competitions/document-image-classification-sd234504 (32003, 4) (7994, 2)


In [3]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".pt"):
            print(os.path.join(root, f))

/kaggle/input/notebooks/azizahadilah/02c-training/checkpoint/efficientnet_b3_best.pt


In [4]:
import os
print(os.listdir("/kaggle/input"))

['notebooks', 'competitions']


In [5]:
import os
for root, dirs, files in os.walk("/kaggle/input/notebooks", followlinks=True):
    print(root, "->", files[:5])

/kaggle/input/notebooks -> []
/kaggle/input/notebooks/azizahadilah -> []
/kaggle/input/notebooks/azizahadilah/02c-training -> ['__huggingface_repos__.json', '__notebook__.ipynb', '__output__.json', '__results__.html', 'custom.css']
/kaggle/input/notebooks/azizahadilah/02c-training/__results___files -> ['__results___8_0.png', '__results___8_2.png']
/kaggle/input/notebooks/azizahadilah/02c-training/checkpoint -> ['efficientnet_b3_best.pt']
/kaggle/input/notebooks/azizahadilah/02c-training/document-image-classification-kaggle -> ['.gitignore', 'README.md', 'requirements.txt']
/kaggle/input/notebooks/azizahadilah/02c-training/document-image-classification-kaggle/.git -> ['HEAD', 'config', 'description', 'index', 'packed-refs']
/kaggle/input/notebooks/azizahadilah/02c-training/document-image-classification-kaggle/.git/hooks -> ['applypatch-msg.sample', 'commit-msg.sample', 'fsmonitor-watchman.sample', 'post-update.sample', 'pre-applypatch.sample']
/kaggle/input/notebooks/azizahadilah/02c-tr

In [6]:
from scripts.preprocess import cache_images
cache_images(test_df, DATA_DIR, CACHE_DIR, IMG_SIZE)
import glob
ckpts = glob.glob("/kaggle/working/checkpoint/*_best.pt") + glob.glob("/kaggle/input/**/*_best.pt", recursive=True)
print(ckpts); CKPT = ckpts[0]

100%|██████████| 7994/7994 [00:20<00:00, 398.05it/s]


Selesai: 7994 berhasil, 0 gagal
['/kaggle/input/notebooks/azizahadilah/02c-training/checkpoint/efficientnet_b3_best.pt']


In [7]:
import torch.nn as nn
from sklearn.model_selection import train_test_split
from scripts.model import build_model
from scripts.dataset import make_loader
from scripts.predict import predict_proba, make_submission
from scripts.preprocess import cache_images
from scripts.train import evaluate

device = "cuda"
print("Checkpoint:", CKPT)
model = build_model("efficientnet_b3", num_classes=train_df["label"].nunique(), pretrained=False)
model.load_state_dict(torch.load(CKPT, map_location="cpu"))
model.to(device)

# Cek: split sama persis dengan di 02_training
_, va_df = train_test_split(train_df, test_size=0.1, stratify=train_df["label"], random_state=42)
cache_images(va_df, DATA_DIR, CACHE_DIR, IMG_SIZE)
val_loader = make_loader(va_df, CACHE_DIR, IMG_SIZE, 32, train=False)
_, acc, _, _ = evaluate(model, val_loader, nn.CrossEntropyLoss(), device)
print("Val acc checkpoint ini pada IMG_SIZE", IMG_SIZE, ":", acc)

Checkpoint: /kaggle/input/notebooks/azizahadilah/02c-training/checkpoint/efficientnet_b3_best.pt


100%|██████████| 3201/3201 [00:07<00:00, 401.84it/s]


Selesai: 3201 berhasil, 0 gagal
Val acc checkpoint ini pada IMG_SIZE 320 : 0.8778506716651047


In [8]:
test_loader = make_loader(test_df, CACHE_DIR, IMG_SIZE, 32, train=False, is_test=True)
probs = predict_proba(model, test_loader, device, tta=True)
os.makedirs("/kaggle/working/submissions", exist_ok=True)
sub = make_submission(test_df["id"].tolist(), probs, f"{DATA_DIR}/sample_submission.csv", "/kaggle/working/submission.csv")
sub.to_csv("/kaggle/working/submissions/submission.csv", index=False)
print(sub.head(), sub.shape, sub["label"].value_counts().sort_index().tolist())

100%|██████████| 250/250 [00:42<00:00,  5.95it/s]

           id  label
0  test_00001      0
1  test_00002      0
2  test_00003      1
3  test_00004      4
4  test_00005     13 (7994, 2) [510, 481, 504, 536, 513, 518, 460, 502, 474, 493, 520, 446, 492, 486, 544, 515]
